# 07 · Extracción canónica de proyectos de generación del BOE · orquestación v25.2

Este notebook extrae exclusivamente la información necesaria para el TFM:

1. **plantas de generación eléctrica** como entidades raíz, agrupables y buscables;
2. **componentes asociados** —almacenamiento y sistema de evacuación/conexión— como entidades secundarias no agrupables;
3. **actuaciones administrativas actuales** con una semántica única de destinatarios;
4. nombres, magnitudes principales, participantes, localizaciones y relaciones materiales necesarias para reconstruir la cronología.

Principio de diseño:

```text
generation_assets = raíces persistentes del proyecto
associated_components = elementos vinculados, nunca raíces de agrupación
targets=["event"] = actuación sobre todo el proyecto del evento
targets concretos = actuación sobre un subconjunto inequívoco
```

La evidencia de una entidad demuestra su existencia. Los enlaces planta–componente y los targets se resuelven con el contexto completo del acto, sin exigir que una única cita contenga simultáneamente toda la estructura.


### Cambio visible de la versión 23

- La clasificación operativa pasa a ser binaria: proyecto de generación específico / no relevante.
- El piloto distingue validación estructural de exactitud de alcance mediante etiquetas externas de evaluación.
- Los anuncios de contratación y los proyectos no energéticos con fotovoltaica auxiliar no se convierten en proyectos de generación.
- Estas reglas no utilizan identificadores BOE y no alteran silenciosamente las extracciones: cada ajuste determinista queda registrado.


**Notebook de orquestación v25.2.** La implementación productiva reside en `renewables_permitting.extraction`; este notebook conserva la orquestación, el piloto y la auditoría. Las garantías de regresión se verifican directamente mediante `tests/extraction`.

**Validación de la semántica de extracción.** La versión v25.2 conserva las invariantes validadas durante el piloto de referencia y actualmente cubiertas por `tests/extraction`. Esta revisión:

- elimina el `FutureWarning` producido al acumular intentos con columnas completamente nulas;
- hace explícita la composición del piloto: **43 documentos específicos** y **57 no relevantes**;
- añade una auditoría tabular de los 100 resultados con nombres de plantas, tecnologías, componentes y actuaciones;
- refuerza la invariancia `sin eventos ⇔ no relevante` sin modificar ninguna extracción ya validada.


In [ ]:
from __future__ import annotations

import json
import re
from hashlib import sha256
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

from renewables_permitting.extraction.paths import (
    ADMINISTRATIVE_ACTIONS_PATH,
    ADMINISTRATIVE_ACTION_TARGETS_PATH,
    ASSOCIATED_COMPONENTS_PATH,
    ASSOCIATED_COMPONENT_GENERATION_LINKS_PATH,
    ASSOCIATED_COMPONENT_NAMES_PATH,
    BOE_AI_EXTRACTIONS_PATH,
    BOE_AI_EXTRACTION_ATTEMPTS_PATH,
    BOE_AI_MANUAL_REVIEWS_PATH,
    BOE_AI_MANUAL_REVIEW_DIR,
    BOE_AI_QUALITY_METRICS_PATH,
    BOE_AI_REVIEW_QUEUE_PATH,
    BOE_CANDIDATES_DOCS_TEXT_PATH,
    CASE_FILE_REFERENCES_PATH,
    DATA_DIR,
    GENERATION_ASSET_MENTIONS_PATH,
    GENERATION_ASSET_NAMES_PATH,
    GENERATION_RELATIONS_PATH,
    LOCATION_MENTIONS_PATH,
    PARTICIPANT_MENTIONS_PATH,
    PROJECT_ROOT,
    PUBLICATION_EVENTS_PATH,
    SILVER_BOE_AI_DIR,
    SILVER_DIR,
    TECHNICAL_MENTIONS_PATH,
    find_project_root,
)
from renewables_permitting.utils import validate_required_columns


## 1. Contrato de extracción

In [ ]:
from renewables_permitting.extraction.models import (
    ActionTargetRef,
    AdministrativeAction,
    AdministrativeActionType,
    AdministrativeDecision,
    AdministrativeLocationLevel,
    AdministrativeLocationMention,
    AssociatedComponent,
    AssociatedComponentType,
    BOEAIExtraction,
    BOEId,
    BOEProjectExtraction,
    BOESourceDocument,
    BOE_ID_ADAPTER,
    ClassificationStatus,
    ComponentRef,
    ContractModel,
    DocumentScope,
    EntityRef,
    GenerationAssetMention,
    GenerationAssetRef,
    GenerationAssetRelation,
    GenerationRelationType,
    GenerationType,
    NonEmptyText,
    ParticipantMention,
    ParticipantRole,
    PreparedDocumentPrompt,
    PublicationEvent,
    SelectedDocumentText,
    TechnicalAttributeType,
    TechnicalMention,
    _ALLOWED_DECISIONS_BY_ACTION_TYPE,
    _canonicalize_refs,
    _deduplicate_strings,
    _reference_sort_key,
    _text_key,
    build_boe_project_extraction,
)


## 2. Instrucciones del agente

In [ ]:
from renewables_permitting.extraction.instructions import (
    AGENT_INSTRUCTIONS,
    CORE_INSTRUCTIONS,
    DECISION_EXAMPLES,
    TAXONOMY_GUIDANCE,
)


## 3. Modelo y configuración reproducible

In [ ]:
from renewables_permitting.extraction.agent import (
    Agent,
    PYDANTIC_AI_AVAILABLE,
    RunUsage,
    UsageLimits,
    build_boe_extraction_agent,
    build_ollama_model,
    validate_runtime_configuration,
)
from renewables_permitting.extraction.config import (
    AGENT_RETRIES,
    AI_MODEL_NAME,
    CHECKPOINT_EVERY,
    COMPONENT_LINK_POLICY,
    CONTRACT_SCHEMA_SHA256,
    DOCUMENT_TIMEOUT_SECONDS,
    DOCUMENT_VALIDATION_RETRY_ATTEMPTS,
    DOCUMENT_VALIDATION_VERSION,
    DOCUMENTARY_MATCH_POLICY,
    ENTITY_MODEL_POLICY,
    EVENT_GRANULARITY_POLICY,
    EXTRACTION_CONFIG,
    EXTRACTION_CONFIG_ID,
    INSTRUCTIONS_SHA256,
    MAX_MODEL_REQUESTS_PER_DOCUMENT,
    MODEL_PROVIDER,
    MODEL_RUN_TIMEOUT_SECONDS,
    MODEL_SETTINGS,
    ModelProvider,
    QUALITY_WORKFLOW_POLICY,
    SCOPE_CLASSIFICATION_POLICY,
    TARGET_SEMANTICS_POLICY,
    TEMPORAL_POLICY,
    TRANSIENT_RETRY_BASE_SECONDS,
    TRANSIENT_RUN_ATTEMPTS,
    USE_NATIVE_OUTPUT,
    _stable_json_hash,
)

AI_MODEL = None
agent = None


## 4. Fuente documental y persistencia

In [ ]:
from renewables_permitting.extraction.documents import (
    DOCUMENT_PROMPT_TEMPLATE,
    MAX_DOCUMENT_CHARS,
    MIN_SUBSTANTIVE_TEXT_CHARS_BEFORE_ANNEX,
    _AFFECTED_ASSETS_ANNEX_HEADING_RE,
    _required_date,
    _required_text,
    _source_document_hash,
    build_document_prompt,
    build_source_document,
    select_document_text,
)
from renewables_permitting.extraction.persistence import save_parquet_atomic


## 5. Canonicalización y validación documental

In [ ]:
from renewables_permitting.extraction.canonicalization import (
    ScopeGuardDecision,
    _ACTION_PATTERNS,
    _AUXILIARY_COMPONENT_TYPES,
    _AUXILIARY_RENEWABLE_RE,
    _COMPONENT_PATTERNS,
    _EXPLICIT_ELECTRIC_GENERATION_RE,
    _GENERATION_DESCRIPTOR_PATTERN,
    _MODIFIABLE_ACTION_TYPES,
    _NON_ELECTRIC_GAS_INFRASTRUCTURE_RE,
    _NON_GENERATION_MAIN_OBJECT_RE,
    _PROCUREMENT_TITLE_RE,
    _STANDALONE_STORAGE_MAIN_OBJECT_RE,
    _STORAGE_LINKED_TO_GENERATION_RE,
    _action_pattern,
    _action_types_from_title,
    _canonical_documentary_text,
    _canonicalize_actions,
    _canonicalize_generation_relations,
    _canonicalize_optional_mentions,
    _component_pattern,
    _component_refs_mentioned,
    _decision_from_title,
    _dedupe_technical_mentions,
    _documentary_contains,
    _entity_refs_mentioned,
    _event_is_integrated,
    _evidence_is_supported,
    _evidence_segments,
    _exact_component_description,
    _expand_multitechnology_generation_assets,
    _find_literal_span,
    _force_non_relevant_extraction,
    _generation_name_is_direct_target,
    _generation_refs_mentioned,
    _infer_action_targets,
    _infer_component_links,
    _infer_generation_type,
    _is_clearly_historical,
    _merge_auxiliary_components,
    _modification_expectation,
    _normalize_action_type,
    _normalize_action_type_from_context,
    _remap_component_targets,
    _renumber_event,
    _repair_action_evidence,
    _repair_component,
    _repair_evidence,
    _repair_generation_asset,
    _repair_technical_mentions,
    _salvage_generation_names,
    _scope_guard_from_document,
    _source_units,
    _split_independent_generation_event,
    canonicalize_project_extraction,
    preclassify_document_without_model,
)
from renewables_permitting.extraction.validation import (
    DocumentExtractionValidationError,
    _generation_name_has_documentary_context,
    build_document_validation_retry_prompt,
    validate_extraction_against_document,
)


## 6. Ejecución, reintentos y registros

In [ ]:
from renewables_permitting.extraction.runner import (
    _REQUIRED_INPUT_COLUMNS,
    _RETRYABLE_HTTP_STATUS_CODES,
    _base_record,
    _is_retryable_model_error,
    _usage_values,
    build_error_record,
    build_success_record,
    debug_state,
    extract_documents,
    get_extraction_model,
    load_and_prepare_candidates,
    run_agent_with_transient_retries,
    run_and_finalize_extractions,
)


## 7. Calidad, revisión manual y selección de la extracción vigente

La extracción automática y la corrección humana se conservan como fuentes
separadas. La tabla vigente se reconstruye de forma determinista: una revisión
manual validada prevalece sobre la extracción automática; una revisión manual
rechazada excluye el documento; en ausencia de revisión se usa la extracción
automática válida más reciente.

La cola de revisión es derivada y no se edita. Cada corrección humana se guarda
como un archivo JSON versionable en `data/manual/boe_ai_reviews/` y se materializa
en `boe_ai_manual_reviews.parquet`.


In [ ]:
from renewables_permitting.extraction.review import (
    AI_EXTRACTION_LOG_COLUMNS,
    MANUAL_REVIEW_COLUMNS,
    QUALITY_METRIC_COLUMNS,
    REVIEW_QUEUE_COLUMNS,
    _count_extracted_nodes,
    _latest_attempts_for_current_sources,
    _latest_manual_reviews_for_current_sources,
    _manual_review_as_extraction_record,
    _normalise_table,
    append_ai_extraction_attempts,
    append_quality_metric,
    build_pending_candidates,
    build_quality_metric,
    build_review_queue,
    combine_ai_extraction_attempt_frames,
    combine_quality_metrics,
    create_manual_review_file,
    current_successful_ai_extractions,
    empty_ai_extraction_attempts_log,
    empty_manual_reviews,
    empty_quality_metrics,
    empty_review_queue,
    load_ai_extraction_attempts,
    load_manual_review_files,
    normalise_ai_extraction_attempts_log,
    normalise_manual_reviews,
    select_best_valid_extractions,
)


## 8. Tablas planas para agrupación y cronología


In [ ]:
from renewables_permitting.extraction.flatten import (
    FLAT_TABLE_COLUMNS,
    flatten_current_extractions,
    save_flattened_extractions,
)


## 9. Suite de regresión determinista


Las regresiones deterministas automatizadas viven en `tests/extraction` y se ejecutan con `pytest`. Este notebook no duplica sus helpers, definiciones ni ejecución.


## 10. Piloto estratificado de 100 documentos: validación estructural y de alcance (opt-in)

El piloto utiliza una muestra estratificada y congelada, independiente de la
versión del prompt. No incorpora revisiones manuales: mide exclusivamente la
calidad automática. Se considera superado únicamente cuando los 100 documentos
tienen un intento vigente, una extracción automática válida y cero fallos de
las invariantes generales.

Los casos conocidos difíciles se controlan en la suite de regresión anterior;
no se fuerzan dentro de la muestra del piloto.


El piloto utiliza un archivo de etiquetas revisadas manualmente exclusivamente para evaluar `document_scope`. Estas etiquetas no se pasan al modelo, no intervienen en la canonicalización y no seleccionan la muestra. Evitan declarar éxito cuando el JSON es válido pero un documento fuera del alcance se ha convertido en proyecto.


### Sobre la celda roja al final del piloto

La instrucción `raise AssertionError(...)` no es un defecto sintáctico. Es una
barrera deliberada: VS Code marca la celda en rojo cuando `test_passed=False`.
No debe eliminarse ni desactivarse para ocultar un fallo. Con una ejecución
correcta, la condición no se cumple y la celda termina sin error.

En la muestra congelada actual se esperan **43 documentos específicos de
proyectos de generación** y **57 documentos no relevantes**. Las instalaciones
autónomas de almacenamiento y los anuncios de contratación se descartan antes
de llamar al modelo.

### Qué demuestra y qué no demuestra el piloto

`test_passed=True` acredita que los 100 documentos:

- tienen una extracción vigente;
- cumplen el contrato Pydantic y las invariantes documentales;
- coinciden con la etiqueta revisada de alcance;
- no dejan plantas sin nombre, eventos sin actuación ni componentes sin vínculo.

No acredita por sí solo que todos los campos opcionales sean exhaustivos. La tabla
`pilot_boe_ai_semantic_audit.parquet` se genera para revisar de forma transparente
los nombres, tecnologías, componentes y actuaciones de los documentos específicos.


In [ ]:

RUN_STRATIFIED_PILOT = False
RESET_PILOT_OUTPUTS = False  # Después de superar el piloto RESET_PILOT_OUTPUTS = False
REBUILD_PILOT_SAMPLE = False
RAISE_ON_PILOT_FAILURE = True

PILOT_SAMPLE_SIZE = 100
PILOT_RANDOM_SEED = 20260720
PILOT_CHECKPOINT_EVERY = 1
PILOT_MIN_AUTO_VALIDATION_RATE = 1.0

PILOT_SAMPLE_PATH = SILVER_BOE_AI_DIR / "pilot_sample_100.parquet"
PILOT_ATTEMPTS_PATH = (
    SILVER_BOE_AI_DIR / "pilot_boe_ai_extraction_attempts.parquet"
)
PILOT_CURRENT_PATH = SILVER_BOE_AI_DIR / "pilot_boe_ai_extractions.parquet"
PILOT_REVIEW_QUEUE_PATH = SILVER_BOE_AI_DIR / "pilot_boe_ai_review_queue.parquet"
PILOT_QUALITY_METRICS_PATH = (
    SILVER_BOE_AI_DIR / "pilot_boe_ai_quality_metrics.parquet"
)
PILOT_SCOPE_LABELS_PATH = SILVER_BOE_AI_DIR / "pilot_scope_labels_100.csv"
PILOT_SCOPE_LABELS_FALLBACK_PATH = Path.cwd() / "pilot_scope_labels_100.csv"
PILOT_SEMANTIC_AUDIT_PATH = (
    SILVER_BOE_AI_DIR / "pilot_boe_ai_semantic_audit.parquet"
)
PILOT_MIN_SCOPE_ACCURACY = 1.0


def load_pilot_scope_labels(
    *,
    expected_sample_ids: set[str],
) -> pd.DataFrame:
    candidate_paths = [
        PILOT_SCOPE_LABELS_PATH,
        PILOT_SCOPE_LABELS_FALLBACK_PATH,
    ]
    path = next((item for item in candidate_paths if item.exists()), None)
    if path is None:
        raise FileNotFoundError(
            "No se encontró pilot_scope_labels_100.csv. Colócalo en "
            f"{PILOT_SCOPE_LABELS_PATH} o en {PILOT_SCOPE_LABELS_FALLBACK_PATH}."
        )
    labels = pd.read_csv(path, dtype={"identificador_boe": "string"})
    validate_required_columns(
        labels,
        {"identificador_boe", "expected_document_scope"},
    )
    if labels["identificador_boe"].duplicated().any():
        raise ValueError("Las etiquetas del piloto contienen BOE duplicados.")
    label_ids = set(labels["identificador_boe"].astype(str))
    if label_ids != expected_sample_ids:
        raise ValueError(
            "Las etiquetas no coinciden exactamente con la muestra congelada. "
            f"faltan={sorted(expected_sample_ids - label_ids)}, "
            f"sobran={sorted(label_ids - expected_sample_ids)}."
        )
    valid_scopes = {item.value for item in DocumentScope}
    invalid = set(labels["expected_document_scope"].dropna().astype(str)) - valid_scopes
    if invalid:
        raise ValueError(f"Etiquetas de alcance inválidas: {sorted(invalid)}")
    return labels.copy()


def _pilot_topic_group(title: str) -> str:
    key = _canonical_documentary_text(title).casefold()
    for group, pattern in [
        ("hybrid_storage", r"hibrid|almacen|bess|bater"),
        ("grid", r"subestaci|l[ií]nea|evacuaci|conexi"),
        ("environmental", r"impacto ambiental|afecci[oó]n ambiental"),
        ("public_utility", r"utilidad p[uú]blica|expropi|ocupaci[oó]n"),
        ("authorizations", r"autorizaci[oó]n administrativa"),
    ]:
        if re.search(pattern, key):
            return group
    return "other"


def build_stratified_pilot_sample(
    source_df: pd.DataFrame,
    *,
    sample_size: int,
) -> pd.DataFrame:
    if sample_size < 1:
        raise ValueError("sample_size debe ser mayor que cero.")
    if sample_size > len(source_df):
        raise ValueError(
            f"La muestra solicitada ({sample_size}) supera el corpus ({len(source_df)})."
        )

    pilot = source_df.copy()
    pilot["source_text_chars"] = pilot["texto_limpio"].astype("string").str.len()
    pilot["publication_year"] = pd.to_datetime(
        pilot["fecha_publicacion"], errors="coerce"
    ).dt.year.astype("Int64")
    pilot["length_band"] = pd.cut(
        pilot["source_text_chars"],
        bins=[0, 20_000, 50_000, 100_000, float("inf")],
        labels=["short", "medium", "long", "very_long"],
        include_lowest=True,
    ).astype("string")
    pilot["topic_group"] = pilot["titulo"].astype(str).map(_pilot_topic_group)
    pilot["stratum"] = (
        pilot["publication_year"].astype("string")
        + "|"
        + pilot["length_band"]
        + "|"
        + pilot["topic_group"]
    )
    # La muestra no cambia cuando cambia el prompt o la versión del validador.
    pilot["sample_priority"] = pilot["identificador"].astype(str).map(
        lambda boe_id: sha256(
            f"{PILOT_RANDOM_SEED}|{boe_id}".encode("utf-8")
        ).hexdigest()
    )

    groups = [
        group.sort_values("sample_priority", kind="stable")
        for _, group in pilot.groupby("stratum", sort=True, dropna=False)
    ]
    selected_indices: list[Any] = []
    for round_index in range(max(len(group) for group in groups)):
        for group in groups:
            if round_index < len(group):
                selected_indices.append(group.index[round_index])
            if len(selected_indices) >= sample_size:
                break
        if len(selected_indices) >= sample_size:
            break

    selected = pilot.loc[selected_indices].copy()
    if len(selected) != sample_size:
        raise AssertionError(
            f"No se pudo construir la muestra completa: {len(selected)} != {sample_size}."
        )
    return selected.reset_index(drop=True)


def load_or_build_pilot_sample(
    source_df: pd.DataFrame,
    *,
    sample_size: int,
    path: Path = PILOT_SAMPLE_PATH,
    rebuild: bool = False,
) -> pd.DataFrame:
    if path.exists() and not rebuild:
        stored = pd.read_parquet(path)
        validate_required_columns(stored, {"identificador"})
        stored_ids = stored["identificador"].astype(str)
        if len(stored_ids) != sample_size or stored_ids.duplicated().any():
            raise ValueError(
                "La muestra congelada no coincide con PILOT_SAMPLE_SIZE. "
                "Usa REBUILD_PILOT_SAMPLE=True para regenerarla deliberadamente."
            )
        missing = set(stored_ids) - set(source_df["identificador"].astype(str))
        if missing:
            raise ValueError(
                f"La muestra congelada contiene BOE ausentes del corpus: {sorted(missing)}"
            )
        order = {boe_id: index for index, boe_id in enumerate(stored_ids)}
        selected = source_df.loc[
            source_df["identificador"].astype(str).isin(order)
        ].copy()
        selected["_pilot_order"] = selected["identificador"].astype(str).map(order)
        return (
            selected.sort_values("_pilot_order", kind="stable")
            .drop(columns="_pilot_order")
            .reset_index(drop=True)
        )

    selected = build_stratified_pilot_sample(
        source_df,
        sample_size=sample_size,
    )
    sample_columns = [
        "identificador",
        "fecha_publicacion",
        "titulo",
        "publication_year",
        "length_band",
        "topic_group",
        "stratum",
        "sample_priority",
    ]
    save_parquet_atomic(selected[sample_columns], path)
    return selected


def evaluate_pilot(
    pilot_sample: pd.DataFrame,
    attempts: pd.DataFrame,
    current: pd.DataFrame,
    scope_labels: pd.DataFrame,
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    selected_ids = set(pilot_sample["identificador"].astype(str))
    current_ids = set(current["identificador_boe"].astype(str))
    latest_attempts = _latest_attempts_for_current_sources(
        attempts,
        pilot_sample,
    )
    latest_ids = set(latest_attempts["identificador_boe"].astype(str))

    source_rows = {
        str(row["identificador"]): row
        for _, row in pilot_sample.iterrows()
    }
    expected_scope_by_id = dict(zip(
        scope_labels["identificador_boe"].astype(str),
        scope_labels["expected_document_scope"].astype(str),
        strict=True,
    ))
    checks: list[dict[str, Any]] = []
    scope_checks: list[dict[str, Any]] = []

    for boe_id in sorted(selected_ids & current_ids):
        extraction = get_extraction_model(current, boe_id)
        predicted_scope = (
            extraction.document_scope.value
            if extraction.document_scope is not None
            else None
        )
        expected_scope = expected_scope_by_id[boe_id]
        generation_assets = [
            asset
            for event in extraction.publication_events
            for asset in event.generation_assets
        ]
        components = [
            component
            for event in extraction.publication_events
            for component in event.associated_components
        ]
        actions = [
            action
            for event in extraction.publication_events
            for action in event.administrative_actions
        ]
        current_row = current.loc[
            current["identificador_boe"].astype(str).eq(boe_id)
        ].iloc[-1]

        scope_checks.append({
            "identificador_boe": boe_id,
            "titulo": str(source_rows[boe_id]["titulo"]),
            "expected_document_scope": expected_scope,
            "predicted_document_scope": predicted_scope,
            "n_publication_events": len(extraction.publication_events),
            "n_generation_assets": len(generation_assets),
            "n_associated_components": len(components),
            "n_administrative_actions": len(actions),
            "generation_asset_names_json": json.dumps(
                [asset.names_raw for asset in generation_assets],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "generation_types_json": json.dumps(
                [asset.generation_type.value for asset in generation_assets],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "component_types_json": json.dumps(
                [component.component_type.value for component in components],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "action_types_json": json.dumps(
                [action.action_type.value for action in actions],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "processing_stage": current_row.get("processing_stage"),
            "classification_reason": extraction.classification_reason,
            "scope_match": predicted_scope == expected_scope,
        })
        try:
            source_document = build_source_document(source_rows[boe_id])
            validate_extraction_against_document(
                document=source_document,
                extraction=extraction,
            )

            for event in extraction.publication_events:
                if not event.generation_assets:
                    raise AssertionError("Evento sin planta de generación.")
                if not event.administrative_actions:
                    raise AssertionError("Evento sin actuación administrativa.")
                if any(not asset.names_raw for asset in event.generation_assets):
                    raise AssertionError("Planta sin nombre documental.")
                if any(
                    not component.related_generation_asset_refs
                    for component in event.associated_components
                ):
                    raise AssertionError("Componente sin vínculo con una planta.")
                if any(
                    not action.targets
                    for action in event.administrative_actions
                ):
                    raise AssertionError("Actuación sin destinatario.")

            passed = True
            reason = None
        except Exception as error:
            passed = False
            reason = str(error)

        checks.append({
            "identificador_boe": boe_id,
            "check": "document_contract_and_tfm_core_invariants",
            "passed": passed,
            "reason": reason,
        })

    checks_df = pd.DataFrame(
        checks,
        columns=["identificador_boe", "check", "passed", "reason"],
    )
    n_errors = int(
        latest_attempts["extraction_status"].ne("ok").fillna(True).sum()
    )
    semantic_failures = (
        int((~checks_df["passed"]).sum())
        if not checks_df.empty
        else len(selected_ids)
    )
    auto_validation_rate = (
        len(current_ids & selected_ids) / len(selected_ids)
        if selected_ids
        else 1.0
    )
    scope_checks_df = pd.DataFrame(scope_checks)
    specific_scope = DocumentScope.GENERATION_PROJECT_SPECIFIC.value
    not_relevant_scope = (
        DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS.value
    )
    scope_event_inconsistency = (
        scope_checks_df["predicted_document_scope"].eq(specific_scope)
        & scope_checks_df["n_publication_events"].eq(0)
    ) | (
        scope_checks_df["predicted_document_scope"].eq(not_relevant_scope)
        & scope_checks_df["n_publication_events"].gt(0)
    )
    n_scope_event_inconsistencies = int(scope_event_inconsistency.sum())

    n_scope_mismatches = (
        int((~scope_checks_df["scope_match"]).sum())
        if not scope_checks_df.empty
        else len(selected_ids)
    )
    scope_accuracy = (
        float(scope_checks_df["scope_match"].mean())
        if not scope_checks_df.empty
        else 0.0
    )
    test_passed = bool(
        len(selected_ids) == PILOT_SAMPLE_SIZE
        and latest_ids == selected_ids
        and current_ids == selected_ids
        and n_errors == 0
        and semantic_failures == 0
        and n_scope_mismatches == 0
        and n_scope_event_inconsistencies == 0
        and auto_validation_rate >= PILOT_MIN_AUTO_VALIDATION_RATE
        and scope_accuracy >= PILOT_MIN_SCOPE_ACCURACY
    )
    summary = {
        "status": "passed" if test_passed else "failed",
        "test_passed": test_passed,
        "n_selected_documents": len(selected_ids),
        "n_latest_attempts": len(latest_ids),
        "n_canonical_extractions": len(current_ids),
        "n_missing_latest_attempts": len(selected_ids - latest_ids),
        "n_missing_canonical_extractions": len(selected_ids - current_ids),
        "n_unexpected_latest_attempts": len(latest_ids - selected_ids),
        "n_unexpected_canonical_extractions": len(current_ids - selected_ids),
        "n_errors": n_errors,
        "n_semantic_failures": semantic_failures,
        "automatic_validation_rate": auto_validation_rate,
        "n_scope_evaluated": len(scope_checks_df),
        "n_scope_mismatches": n_scope_mismatches,
        "scope_accuracy": scope_accuracy,
        "n_scope_event_inconsistencies": n_scope_event_inconsistencies,
        "n_zero_event_documents": int(
            scope_checks_df["n_publication_events"].eq(0).sum()
        ),
        "n_project_specific_documents": int(
            scope_checks_df["predicted_document_scope"].eq(
                DocumentScope.GENERATION_PROJECT_SPECIFIC.value
            ).sum()
        ),
        "n_not_relevant_documents": int(
            scope_checks_df["predicted_document_scope"].eq(
                DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS.value
            ).sum()
        ),
        "n_total_publication_events": int(
            scope_checks_df["n_publication_events"].sum()
        ),
        "n_total_generation_assets": int(
            scope_checks_df["n_generation_assets"].sum()
        ),
        "n_total_associated_components": int(
            scope_checks_df["n_associated_components"].sum()
        ),
        "n_total_administrative_actions": int(
            scope_checks_df["n_administrative_actions"].sum()
        ),
        "n_deterministic_scope_guards": int(
            scope_checks_df["processing_stage"]
            .eq("deterministic_scope_guard")
            .sum()
        ),
    }
    return summary, checks_df, scope_checks_df


if RUN_STRATIFIED_PILOT:
    if agent is None:
        validate_runtime_configuration()

        if MODEL_PROVIDER == "gemini":
            AI_MODEL = AI_MODEL_NAME
        elif MODEL_PROVIDER == "ollama":
            AI_MODEL = build_ollama_model(AI_MODEL_NAME)
        else:
            raise ValueError(
                "MODEL_PROVIDER debe ser 'gemini' u 'ollama': "
                f"{MODEL_PROVIDER!r}."
            )

        agent = build_boe_extraction_agent()

    candidates = load_and_prepare_candidates()
    pilot_sample = load_or_build_pilot_sample(
        candidates,
        sample_size=PILOT_SAMPLE_SIZE,
        rebuild=REBUILD_PILOT_SAMPLE,
    )

    pilot_scope_labels = load_pilot_scope_labels(
        expected_sample_ids=set(pilot_sample["identificador"].astype(str)),
    )

    if RESET_PILOT_OUTPUTS:
        for path in (
            PILOT_ATTEMPTS_PATH,
            PILOT_CURRENT_PATH,
            PILOT_REVIEW_QUEUE_PATH,
        ):
            path.unlink(missing_ok=True)

    attempts = load_ai_extraction_attempts(PILOT_ATTEMPTS_PATH)
    pilot_manual_reviews = empty_manual_reviews()
    current, pending = build_pending_candidates(
        pilot_sample,
        attempts,
        manual_reviews=pilot_manual_reviews,
    )
    pilot_result = await run_and_finalize_extractions(
        pending,
        pilot_sample,
        agent=agent,
        attempts_path=PILOT_ATTEMPTS_PATH,
        current_path=PILOT_CURRENT_PATH,
        review_queue_path=PILOT_REVIEW_QUEUE_PATH,
        quality_metrics_path=PILOT_QUALITY_METRICS_PATH,
        manual_reviews=pilot_manual_reviews,
        run_scope="pilot",
        minimum_auto_validation_rate=PILOT_MIN_AUTO_VALIDATION_RATE,
        checkpoint_every=PILOT_CHECKPOINT_EVERY,
    )
    pilot_summary, pilot_checks, pilot_scope_checks = evaluate_pilot(
        pilot_sample,
        pilot_result["all_attempts"],
        pilot_result["current_extractions"],
        pilot_scope_labels,
    )

    pilot_quality_metric = pilot_result["quality_metric"].copy()
    pilot_quality_metric["n_scope_evaluated"] = pilot_summary["n_scope_evaluated"]
    pilot_quality_metric["n_scope_mismatches"] = pilot_summary["n_scope_mismatches"]
    pilot_quality_metric["scope_accuracy"] = pilot_summary["scope_accuracy"]
    pilot_quality_metric["minimum_scope_accuracy"] = PILOT_MIN_SCOPE_ACCURACY
    pilot_quality_metric["quality_alert"] = (
        pilot_quality_metric["quality_alert"].fillna(False)
        | (pilot_summary["scope_accuracy"] < PILOT_MIN_SCOPE_ACCURACY)
    )
    pilot_quality_metric["quality_status"] = pilot_quality_metric["quality_alert"].map(
        {True: "degraded", False: "healthy"}
    )
    append_quality_metric(pilot_quality_metric, PILOT_QUALITY_METRICS_PATH)

    save_parquet_atomic(
        pilot_scope_checks,
        PILOT_SEMANTIC_AUDIT_PATH,
    )

    expected_distribution = (
        pilot_scope_checks["expected_document_scope"]
        .value_counts(dropna=False)
        .rename("expected_count")
    )
    predicted_distribution = (
        pilot_scope_checks["predicted_document_scope"]
        .value_counts(dropna=False)
        .rename("predicted_count")
    )
    pilot_scope_distribution = (
        pd.concat(
            [expected_distribution, predicted_distribution],
            axis=1,
        )
        .fillna(0)
        .astype(int)
        .rename_axis("document_scope")
        .reset_index()
    )

    print(
        "Composición del piloto: "
        f"{pilot_summary['n_project_specific_documents']} documentos "
        "específicos de proyectos y "
        f"{pilot_summary['n_not_relevant_documents']} no relevantes."
    )
    display(pd.DataFrame([pilot_summary]))
    display(pilot_scope_distribution)

    print("Fallos de invariantes documentales o del contrato:")
    display(pilot_checks.loc[~pilot_checks["passed"]])

    print("Discordancias respecto de las etiquetas de alcance:")
    display(pilot_scope_checks.loc[~pilot_scope_checks["scope_match"]])

    print("Documentos específicos que alimentarán agrupamiento y cronología:")
    display(
        pilot_scope_checks.loc[
            pilot_scope_checks["predicted_document_scope"].eq(
                DocumentScope.GENERATION_PROJECT_SPECIFIC.value
            ),
            [
                "identificador_boe",
                "titulo",
                "n_publication_events",
                "n_generation_assets",
                "n_associated_components",
                "n_administrative_actions",
                "generation_asset_names_json",
                "generation_types_json",
                "action_types_json",
                "processing_stage",
            ],
        ]
    )

    print(
        "Documentos no relevantes: por contrato deben tener cero eventos. "
        "Esta tabla no representa los 100 documentos del piloto."
    )
    display(
        pilot_scope_checks.loc[
            pilot_scope_checks["predicted_document_scope"].eq(
                DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS.value
            ),
            [
                "identificador_boe",
                "titulo",
                "classification_reason",
                "processing_stage",
                "scope_match",
            ],
        ]
    )
    display(pilot_result["review_queue"])
    display(pilot_quality_metric)

    pilot_passed = bool(pilot_summary.get("test_passed", False))
    if RAISE_ON_PILOT_FAILURE and not pilot_passed:
        failed_dimensions = {
            key: pilot_summary[key]
            for key in (
                "n_errors",
                "n_semantic_failures",
                "n_scope_mismatches",
                "n_scope_event_inconsistencies",
                "automatic_validation_rate",
                "scope_accuracy",
            )
        }
        raise AssertionError(
            "El piloto estratificado no se considera superado. "
            f"Controles fallidos: {failed_dimensions}. "
            f"Resumen completo: {pilot_summary}"
        )


## 11. Producción, revisión manual y regeneración de tablas (opt-in)

La producción no se bloquea por documentos excepcionales. Los intentos no
validados se escriben en `boe_ai_review_queue.parquet`. Para corregir uno:

1. ejecuta `create_manual_review_file(review_queue, "BOE-...")`;
2. edita el JSON creado en `data/manual/boe_ai_reviews/`;
3. cambia `review_status` a `manually_validated`, indica `reviewer` y corrige
   `corrected_extraction`;
4. ejecuta esta sección con `REFRESH_REVIEW_WORKFLOW=True`.

El JSON se valida con el mismo contrato y contra el BOE. Después, la revisión
manual se integra automáticamente con las extracciones automáticas válidas.


### Interpretación del piloto

`OK events=0` significa únicamente que la salida cumple el contrato como documento no relevante. Desde la versión 23 el piloto también compara esa decisión con `pilot_scope_labels_100.csv`. El piloto solo se considera superado si la exactitud de alcance y la validación estructural son ambas del 100 %.


Los descartes deterministas de alta precisión (contratación pública, infraestructura gasista y almacenamiento autónomo sin planta asociada) se guardan como extracciones válidas con `processing_stage=deterministic_scope_guard` y no pasan por la cola de revisión.


In [ ]:

RUN_PRODUCTION_EXTRACTION = False
RESET_PRODUCTION_OUTPUTS = False
RESET_QUALITY_METRICS = False
REFRESH_REVIEW_WORKFLOW = False
REGENERATE_FLAT_TABLES = False

PRODUCTION_BOE_IDS: tuple[str, ...] = ()
PRODUCTION_MAX_DOCUMENTS: int | None = None
PRODUCTION_MIN_AUTO_VALIDATION_RATE = 0.95


if RUN_PRODUCTION_EXTRACTION or REFRESH_REVIEW_WORKFLOW:
    all_candidates = load_and_prepare_candidates()

    if RESET_PRODUCTION_OUTPUTS:
        for path in (
            BOE_AI_EXTRACTION_ATTEMPTS_PATH,
            BOE_AI_EXTRACTIONS_PATH,
            BOE_AI_REVIEW_QUEUE_PATH,
        ):
            path.unlink(missing_ok=True)
    if RESET_QUALITY_METRICS:
        BOE_AI_QUALITY_METRICS_PATH.unlink(missing_ok=True)

    manual_reviews = load_manual_review_files(
        all_candidates,
        review_dir=BOE_AI_MANUAL_REVIEW_DIR,
        output_path=BOE_AI_MANUAL_REVIEWS_PATH,
    )
    attempts = load_ai_extraction_attempts()

    run_candidates = all_candidates.copy()
    if PRODUCTION_BOE_IDS:
        run_candidates = run_candidates.loc[
            run_candidates["identificador"].astype(str).isin(PRODUCTION_BOE_IDS)
        ].copy()
    if PRODUCTION_MAX_DOCUMENTS is not None:
        run_candidates = run_candidates.head(PRODUCTION_MAX_DOCUMENTS).copy()

    if RUN_PRODUCTION_EXTRACTION:
        if agent is None:
            validate_runtime_configuration()

            if MODEL_PROVIDER == "gemini":
                AI_MODEL = AI_MODEL_NAME
            elif MODEL_PROVIDER == "ollama":
                AI_MODEL = build_ollama_model(AI_MODEL_NAME)
            else:
                raise ValueError(
                    "MODEL_PROVIDER debe ser 'gemini' u 'ollama': "
                    f"{MODEL_PROVIDER!r}."
                )

            agent = build_boe_extraction_agent()

        _, pending = build_pending_candidates(
            run_candidates,
            attempts,
            manual_reviews=manual_reviews,
        )
        production_result = await run_and_finalize_extractions(
            pending,
            all_candidates,
            agent=agent,
            attempts_path=BOE_AI_EXTRACTION_ATTEMPTS_PATH,
            current_path=BOE_AI_EXTRACTIONS_PATH,
            review_queue_path=BOE_AI_REVIEW_QUEUE_PATH,
            quality_metrics_path=BOE_AI_QUALITY_METRICS_PATH,
            manual_reviews=manual_reviews,
            run_scope="production",
            minimum_auto_validation_rate=PRODUCTION_MIN_AUTO_VALIDATION_RATE,
            checkpoint_every=CHECKPOINT_EVERY,
        )
        attempts = production_result["all_attempts"]
        current_extractions = production_result["current_extractions"]
        review_queue = production_result["review_queue"]
        quality_metric = production_result["quality_metric"]
    else:
        current_extractions = select_best_valid_extractions(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
        )
        save_parquet_atomic(current_extractions, BOE_AI_EXTRACTIONS_PATH)

        review_queue = build_review_queue(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
        )
        save_parquet_atomic(review_queue, BOE_AI_REVIEW_QUEUE_PATH)

        quality_metric = build_quality_metric(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
            run_scope="production",
            minimum_auto_validation_rate=PRODUCTION_MIN_AUTO_VALIDATION_RATE,
        )
        append_quality_metric(
            quality_metric,
            BOE_AI_QUALITY_METRICS_PATH,
        )

    display(current_extractions[
        [
            "identificador_boe",
            "selection_source",
            "document_scope",
            "n_publication_events",
            "n_generation_assets",
            "n_associated_components",
            "n_administrative_actions",
        ]
    ])
    display(review_queue)
    display(quality_metric)


if REGENERATE_FLAT_TABLES:
    current_extractions = normalise_ai_extraction_attempts_log(
        pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)
    )
    flattened_tables = save_flattened_extractions(current_extractions)
    for table_name, dataframe in flattened_tables.items():
        print(f"{table_name}: {len(dataframe):,} filas")
